In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
import default_risk.config as cfg
import logging
import dtale
import gc

log = logging.getLogger('werkzeug')



cash_balance_df = pd.read_csv(cfg.POS_CASH_BALANCE)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

In [ ]:
#files for the data dictionary
create_files_nulls_per_colmun(cash_balance_df,"POS_CASH_balance")

In [ ]:
#run the screening script on POS_CASH_balance
eda_per_table_printing_results(cash_balance_df, schema, "POS_CASH_balance",False)

In [ ]:
previous_contracts_id_with_nulls=cash_balance_df[cash_balance_df["CNT_INSTALMENT"].isnull()]["SK_ID_PREV"]
rows_of_contracts_with_null=cash_balance_df[cash_balance_df["SK_ID_PREV"].isin(previous_contracts_id_with_nulls)]
len(rows_of_contracts_with_null)

In [ ]:
#how the series looks like without seleccion biases (limitating to 1000 first to handle better the visualization)
first_ids_prev_app= (cash_balance_df["SK_ID_PREV"].unique())[:1000]
slice_of_applications=cash_balance_df[cash_balance_df["SK_ID_PREV"].isin(first_ids_prev_app)]
dtale.show(slice_of_applications.sort_values(["SK_ID_PREV","MONTHS_BALANCE"]))

In [ ]:
#in order to understand the nature of the missing values in this table we will visualize the series with nulls
previous_contracts_id_with_nulls=cash_balance_df[cash_balance_df["CNT_INSTALMENT"].isnull()]["SK_ID_PREV"].unique()
rows_of_contracts_with_null=cash_balance_df[cash_balance_df["SK_ID_PREV"].isin(previous_contracts_id_with_nulls)]
sorted_list=rows_of_contracts_with_null.sort_values(["SK_ID_PREV","MONTHS_BALANCE"])
dtale.show(sorted_list)
#this show the correlation between missing in CNT_INSTALMENT and the status of the contract. CNT_INSTALLMENT seems to be the expected amount total of installment. Sometimes we can see
#how in the row of the secuence that mark the loan as "completed" (NAME_CONTRACT_STATUS == "Completed") the field change to the actual duration of the loan. Meaning the client made payments in advance
#and also we can see the increment pointing to a reschedule of the total duration of the loan. 


In [ ]:
states_with_no_nulls=["Active","Completed"]
inconsistency_status_mask=cash_balance_df["CNT_INSTALMENT"].isnull()  &  cash_balance_df["NAME_CONTRACT_STATUS"].isin(states_with_no_nulls)
cash_balance_df[inconsistency_status_mask]
#this type of nulls (in active contracts) are candidates to be filled 

In [ ]:
#now lets visualize the entire series that have at least one row with status "Active - Complete" and CNT_INSTALMENT in null.
previous_contracts_id_with_nulls=cash_balance_df[inconsistency_status_mask]["SK_ID_PREV"].unique()
rows_of_contracts_with_null=cash_balance_df[cash_balance_df["SK_ID_PREV"].isin(previous_contracts_id_with_nulls)]
sorted_list_with_non_expected_nulls=rows_of_contracts_with_null.sort_values(["SK_ID_PREV","MONTHS_BALANCE"])
dtale.show(sorted_list_with_non_expected_nulls)
#from here we can deduce a fill rule for this non exepecteable nulls. 1- if the sequence have an row with active and "CNT_INSTALMENT" == NULL, often are the first one, a likely registration error easy
#to correct. Putting the status as "Signed" at that point following the pattern that the rest applications folow in this table. But if have more than one row of the secuence with "CNT_INSTALMENT" == NULL can
#assume is prove of data corruption or a canceled contract, someting we want to ignore in the agg metrics.

In [ ]:
#Based on the last visualizations, we detect a very intersting case that could be useful as heuristic to detect series with anormal behaivor. When the "CNT_INSTALMENT_FUTURE" is 0 more than one time, because
#this only should happend at the moment the contract is marked as "Completed"
ceros_df = cash_balance_df.groupby("SK_ID_PREV").filter(lambda g: (g["CNT_INSTALMENT_FUTURE"] == 0).sum() > 1)
ceros_df.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"], inplace=True)
pd.set_option('display.max_columns', None)
dtale.show(ceros_df)

#this help us to detect the "double countability" of the final month. Sometimes they put "Active" with "CNT_INSTALMENT_FUTURE" == 0 and just after another row marking the loan as "Completed"
#and have no day past due. That's mean they are countabilizating twice the final row with the row "active" and "completed" with "CNT_INSTALMENT_FUTURE" == 0. Now we want to analize excluding those cases
#that are easy to correct.

In [ ]:
#Series with "CNT_INSTALMENT_FUTURE" == 0 in more than one row, without counting the "Completed" row, because is the "excpectable" 0 of the serie. This avoid the "Double countability" just detected
#in the last vizualization.
no_completed_status_ceros_df = cash_balance_df.groupby("SK_ID_PREV").filter(lambda g: ((g["CNT_INSTALMENT_FUTURE"] == 0)&(g["NAME_CONTRACT_STATUS"] != "Completed")).sum() > 1)
no_completed_status_ceros_df.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"], inplace=True)
pd.set_option('display.max_columns', None)
dtale.show(no_completed_status_ceros_df)
#this visualization is very useful to recognize two things 1- the "deat tails" of some loans, where  te "CNT_INSTALMENT_FUTURE" is 0, there is any change in any relevant variable but the loan have more rows after that,
#so we can't see any activity but the loan still having more observations, with the status of "active" or "completed".  We will chatch this with a feature but discard it for the agg metrics. 2-Some corrupted data
#of CNT_INSTALMENT_FUTURE being 0 across all the temporal serie. We will desing the features to differences this two type of cases.

In [ ]:
counter_of_unique_values = cash_balance_df.groupby("SK_ID_PREV")["CNT_INSTALMENT_FUTURE"].nunique()
ids_application_constant_value= counter_of_unique_values[counter_of_unique_values == 1].index
id_mask = cash_balance_df["SK_ID_PREV"].isin(ids_application_constant_value)
series_with_constant_value= cash_balance_df[id_mask].copy()
series_with_constant_value.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"], inplace=True)
del id_mask, counter_of_unique_values
gc.collect()
dtale.show(series_with_constant_value)
#Excluding records with a single row (which typically represent recent applications), we can observe a numerous instances of stagnant counters. This suggests to be corrupted data based on CNT_INSTALMENT_FUTURE
#expectable behavior (monotonically decreasing).
#This can be generalizated further as a rule if CNT_INSTALMENT_FUTURE must to contain a minumun ammout of unique values determinated by the length of the series.
#For instance: A loan originally planned for 10 months with 10 rows, needs a minimum of 10 different states. 
#Also we can extend this for edge cases.
#If the loan was planned for 12 months and have 4 rows, need at least 4 unique values (paid in advance).
#and if the Loan was originally planned for 10 months and last more than that (rescheduling of the debt) we expect at least 10 different values in that field. 

In [ ]:
searched_status= ["Approved","Singed"]
status_mask=cash_balance_df["NAME_CONTRACT_STATUS"].isin(searched_status)
id_contracts_with_status=cash_balance_df[status_mask]["SK_ID_PREV"].unique()

print(len(id_contracts_with_status))

print(len(cash_balance_df["SK_ID_PREV"].unique()))  


In [ ]:
len(ceros_df["SK_ID_PREV"].unique())


In [ ]:
#Another think that seems very useful is the ammount of changes in "CNT_INSTALMENT". How is the "expected" total amount of installment at that point of the loan, means a replanification of the loan
#o inconsistency (there is series where this number change constanly, a clear sign of data corruption)

ceros_in_expected_instalment = cash_balance_df.groupby("SK_ID_PREV").filter(lambda g: (len(g["CNT_INSTALMENT"].unique()))> 3)
ceros_in_expected_instalment.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"], inplace=True)
dtale.show(ceros_in_expected_instalment)


In [ ]:
mask = cash_balance_df["SK_ID_PREV"] == 1000033

(cash_balance_df.loc[mask, "CNT_INSTALMENT_FUTURE"] == 0).sum()

In [ ]:
(cash_balance_df["SK_DPD_DEF"] > 10).sum()